[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Leonardo-Baptista/aula_SAM_propemm/blob/main/notebooks/01_sam1_guiado.ipynb)

No computador, use o kernel do `.venv` com `scripts/instalar.py --notebooks` (instala jupyter, matplotlib e segment-anything). O peso ViT-B (375 MB) e a foto são baixados para `sam_aula_cache/` na primeira execução.


# Aula prática: do conjunto de máscaras ao objeto indicado

Notebook didático original para a aula de Leonardo Gonçalves Batista. A sequência de perguntas foi inspirada pelo [tutorial da Roboflow](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/how-to-segment-anything-with-sam.ipynb#scrollTo=pi3C4uDWo10h). A API e a imagem vêm do [repositório oficial SAM 1](https://github.com/facebookresearch/segment-anything). Leia também [Kirillov et al. (2023)](https://arxiv.org/abs/2304.02643).

**Objetivo:** observar a diferença entre gerar regiões candidatas automaticamente e indicar um objeto com caixa ou pontos. O modelo escolhido é ViT-B para reduzir o tamanho do checkpoint. Não há garantia de tempo de execução em CPU. Execute a instalação e o download antes da aula.


In [ ]:
%pip -q install "git+https://github.com/facebookresearch/segment-anything.git" numpy pillow matplotlib


In [ ]:
from pathlib import Path
import time
import urllib.request
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE, "| PyTorch:", torch.__version__)


## 1. Baixar o checkpoint e a fotografia

O checkpoint ViT-B e a fotografia são distribuídos pelo projeto oficial. A primeira execução baixa centenas de megabytes. É melhor preparar esta etapa antes de entrar na sala.


In [ ]:
cache = Path("sam_aula_cache")
cache.mkdir(exist_ok=True)
weights = cache / "sam_vit_b_01ec64.pth"
photo = cache / "dog.jpg"
sources = {
    weights: "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    photo: "https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg",
}
for destination, url in sources.items():
    if not destination.exists():
        print("Baixando", destination.name)
        urllib.request.urlretrieve(url, destination)
    print(destination.name, round(destination.stat().st_size / 1_000_000, 1), "MB")


In [ ]:
start = time.perf_counter()
sam = sam_model_registry["vit_b"](checkpoint=str(weights))
sam.to(device=DEVICE).eval()
print("Carregamento:", round(time.perf_counter()-start, 1), "s")
image = np.asarray(Image.open(photo).convert("RGB"))
print("Imagem:", image.shape, "(altura, largura, canais)")
plt.figure(figsize=(10,6)); plt.imshow(image); plt.axis("off"); plt.show()


## 2. Geração automática

Pergunta para a turma: “Se não dei o nome nem a posição de um objeto, o que exatamente o modelo pode devolver?” A geração automática amostra pontos em uma grade e devolve várias regiões candidatas. Em CPU, usamos uma grade menor para reduzir o trabalho; isso também pode mudar o número de regiões encontradas.


In [ ]:
point_grid = 16 if DEVICE == "cuda" else 8
generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=point_grid,
    points_per_batch=16,
    crop_n_layers=0,
)
start = time.perf_counter()
records = generator.generate(image)
print("Tempo:", round(time.perf_counter()-start,1), "s | máscaras:",len(records))


In [ ]:
overlay = image.astype(np.float32).copy()
palette = np.random.default_rng(7)
for record in sorted(records,key=lambda item:item["area"],reverse=True):
    color = palette.integers(0,256,3)
    mask = record["segmentation"]
    overlay[mask] = 0.55*overlay[mask] + 0.45*color
fig,axes=plt.subplots(1,2,figsize=(14,6))
axes[0].imshow(image); axes[0].set_title("Entrada")
axes[1].imshow(overlay.astype(np.uint8)); axes[1].set_title("Regiões candidatas")
for ax in axes: ax.axis("off")
plt.show()


In [ ]:
if records:
    example = max(records,key=lambda item:item["area"])
    print("Campos:",list(example))
    print("Máscara:",example["segmentation"].shape,"| imagem:",image.shape[:2])
    print("Área:",example["area"],"pixels | caixa XYWH:",example["bbox"])
    print("IoU prevista pelo modelo:",round(example["predicted_iou"],3))


**Discussão:** `segmentation` é uma matriz na ordem altura × largura. `predicted_iou` é uma estimativa do próprio modelo, não uma IoU medida contra uma anotação de referência. Por que apareceram várias regiões, e como escolheríamos a que realmente interessa? [Código oficial da geração automática](https://github.com/facebookresearch/segment-anything/blob/main/segment_anything/automatic_mask_generator.py).


## 3. Uma caixa muda a pergunta

Agora a intenção é explícita: uma caixa aproximadamente ao redor do cão. A caixa usa `[x_min, y_min, x_max, y_max]`. Peça à turma para prever o efeito de deslocar uma borda até incluir o fundo ou cortar parte do animal.


In [ ]:
predictor = SamPredictor(sam)
start = time.perf_counter()
predictor.set_image(image)
print("Codificação da imagem:",round(time.perf_counter()-start,1),"s")
box = np.array([5,175,410,533])
masks,scores,logits = predictor.predict(box=box,multimask_output=True)
print("Máscaras:",masks.shape,"| pontuações:",np.round(scores,3))


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(18,5))
axes[0].imshow(image)
axes[0].add_patch(Rectangle((box[0],box[1]),box[2]-box[0],box[3]-box[1],fill=False,edgecolor="red",linewidth=2))
axes[0].set_title("Caixa indicada")
for i,ax in enumerate(axes[1:]):
    ax.imshow(image)
    ax.imshow(np.ma.masked_where(~masks[i],masks[i]),alpha=.55,cmap="spring")
    ax.set_title(f"Hipótese {i+1}: {scores[i]:.3f}")
for ax in axes: ax.axis("off")
plt.show()


**Experimento da turma:** altere uma coordenada da caixa, execute as duas células anteriores e compare. Depois, discuta por que três máscaras podem ser plausíveis para o mesmo prompt. A métrica decisiva para um projeto requer máscara de referência e critérios do domínio.


## 4. Pontos positivos e negativos, extensão

Este bloco cabe se a demonstração por caixa terminar cedo. Os pontos são apenas sugestões para a fotografia fornecida; ajuste as coordenadas após olhar a imagem.


In [ ]:
points = np.array([[150,390],[610,240]])
labels = np.array([1,0])
point_masks,point_scores,_ = predictor.predict(
    point_coords=points,point_labels=labels,multimask_output=True
)
best = int(np.argmax(point_scores))
plt.figure(figsize=(9,6)); plt.imshow(image)
plt.imshow(np.ma.masked_where(~point_masks[best],point_masks[best]),alpha=.5,cmap="winter")
plt.scatter(points[:,0],points[:,1],c=["lime","red"],edgecolors="black",s=150)
plt.title("Verde: incluir | vermelho: excluir")
plt.axis("off"); plt.show()


**Síntese:** geração automática pergunta “quais regiões há aqui?”. Caixa e pontos perguntam “qual região eu quero?”. Nenhuma dessas chamadas do SAM 1 atribui uma classe semântica confiável por si só. O próximo passo do curso é o SAM 2, que acompanha a região ao longo do vídeo, e o SAM 3, que aceita o conceito como prompt.
